In [ ]:
# ── Cell 1: Mount Drive ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ── Cell 2: Config ─────────────────────────────────────────────────────────
from pathlib import Path
import torch

# ── Paths ──────────────────────────────────────────────────────────────────
SAVE_PATH  = Path("/content/drive/MyDrive/SER_Project")
PICKLE_SRC = str(SAVE_PATH / "src_audio_4s.pkl")
PICKLE_TGT = str(SAVE_PATH / "tgt_audio_4s.pkl")

# ── Audio ──────────────────────────────────────────────────────────────────
SAMPLE_RATE   = 16000
DURATION      = 4.5
TARGET_LENGTH = int(SAMPLE_RATE * DURATION)

# ── Training ───────────────────────────────────────────────────────────────
BATCH_SIZE      = 16
EPOCHS          = 50
PATIENCE        = 10
SEED            = 42
LR_WAVLM        = 1e-5
LR_HEAD         = 1e-3
LAMBDA_SUPCON   = 0.1
LAMBDA_SIMCLR   = 0.1
SUPCON_TEMP     = 0.07
SIMCLR_TEMP     = 0.07
KEEP_RATE       = 0.87
UNFREEZE_N      = 6
PROJ_DIM        = 128    # projection head output dim

# ── Dataset ────────────────────────────────────────────────────────────────
NUM_CLASSES  = 7
CLASS_LABELS = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]

# ── Device ─────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"GPU    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

Device : cuda
GPU    : NVIDIA A100-SXM4-40GB


In [ ]:
# ── Cell 3: Imports & Helpers ──────────────────────────────────────────────
import os, random, warnings, pickle, json
warnings.filterwarnings("ignore")

import numpy as np
import librosa
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from transformers import WavLMModel, Wav2Vec2FeatureExtractor

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, \
                            ConfusionMatrixDisplay, f1_score, \
                            precision_score, recall_score

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# ── Seeds ──────────────────────────────────────────────────────────────────
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

# ── Label encoder ──────────────────────────────────────────────────────────
le = LabelEncoder()
le.fit(CLASS_LABELS)

# ── Dataset class ──────────────────────────────────────────────────────────
class SERDataset(Dataset):
    def __init__(self, audio_list, label_list, feature_extractor):
        self.audio             = audio_list
        self.labels            = torch.tensor(le.transform(label_list), dtype=torch.long)
        self.feature_extractor = feature_extractor

    def __len__(self):
        return len(self.audio)

    def __getitem__(self, idx):
        inputs = self.feature_extractor(
            self.audio[idx],
            sampling_rate=SAMPLE_RATE,
            return_tensors="pt",
            padding=True
        )
        return inputs.input_values.squeeze(0), self.labels[idx]


# ── Target dataset with augmentation for SimCLR ────────────────────────────
class SERDatasetSimCLR(Dataset):
    """
    Returns two augmented views of the same audio for SimCLR.
    No labels used — purely self-supervised.
    """
    def __init__(self, audio_list, feature_extractor):
        self.audio             = audio_list
        self.feature_extractor = feature_extractor

    def __len__(self):
        return len(self.audio)

    def augment(self, y):
        aug_type = random.randint(0, 2)
        if aug_type == 0:
            noise = np.random.randn(len(y)) * 0.005
            return np.clip(y + noise, -1, 1)
        elif aug_type == 1:
            n_steps = random.choice([-2, -1, 1, 2])
            return librosa.effects.pitch_shift(y, sr=SAMPLE_RATE, n_steps=n_steps)
        else:
            rate = random.choice([0.9, 1.1])
            aug = librosa.effects.time_stretch(y, rate=rate)
            if len(aug) < TARGET_LENGTH:
                aug = np.pad(aug, (0, TARGET_LENGTH - len(aug)))
            else:
                aug = aug[:TARGET_LENGTH]
            return aug

    def __getitem__(self, idx):
        y = self.audio[idx]
        view1 = self.feature_extractor(
            self.augment(y), sampling_rate=SAMPLE_RATE,
            return_tensors="pt", padding=True
        ).input_values.squeeze(0)
        view2 = self.feature_extractor(
            self.augment(y), sampling_rate=SAMPLE_RATE,
            return_tensors="pt", padding=True
        ).input_values.squeeze(0)
        return view1, view2


# ── WavLM processor ────────────────────────────────────────────────────────
print("Loading WavLM processor...")
wavlm_processor = Wav2Vec2FeatureExtractor.from_pretrained(
    "microsoft/wavlm-large"
)
print("All imports and helpers ready.")

Loading WavLM processor...


preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

All imports and helpers ready.


In [ ]:
# ── Cell 4: Data Loading ───────────────────────────────────────────────────
print("Loading from pickle...")
with open(PICKLE_SRC, "rb") as f:
    src_audio, src_labels = pickle.load(f)
with open(PICKLE_TGT, "rb") as f:
    tgt_audio, tgt_labels = pickle.load(f)
print(f"Source: {len(src_audio)}, Target: {len(tgt_audio)}")

# Split TESS
tgt_labels_enc = le.transform(tgt_labels)
tgt_adapt_audio, tgt_test_audio, tgt_adapt_labels, tgt_test_labels = train_test_split(
    tgt_audio, tgt_labels,
    test_size=0.2, random_state=SEED,
    stratify=tgt_labels_enc
)

# Standard datasets
src_dataset      = SERDataset(src_audio,       src_labels,       wavlm_processor)
tgt_test_dataset = SERDataset(tgt_test_audio,  tgt_test_labels,  wavlm_processor)

# SimCLR dataset for target adapt (no labels)
tgt_simclr_dataset = SERDatasetSimCLR(tgt_adapt_audio, wavlm_processor)

src_loader       = DataLoader(src_dataset,        batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
tgt_test_loader  = DataLoader(tgt_test_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
tgt_simclr_loader = DataLoader(tgt_simclr_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)

print(f"\nSource       : {len(src_audio)} samples, {len(src_loader)} batches")
print(f"Target SimCLR: {len(tgt_adapt_audio)} samples, {len(tgt_simclr_loader)} batches")
print(f"Target test  : {len(tgt_test_audio)} samples, {len(tgt_test_loader)} batches")

Loading from pickle...
Source: 7488, Target: 5600

Source       : 7488 samples, 468 batches
Target SimCLR: 4480 samples, 280 batches
Target test  : 1120 samples, 70 batches


In [ ]:
# ── Cell 5: DPPMI + SupCon + SimCLR ───────────────────────────────────────
class DPPMIDropout(nn.Module):
    def __init__(self, keep_rate=0.7):
        super().__init__()
        self.keep_rate = keep_rate

    def forward(self, x):
        if not self.training:
            return x
        mean      = x.mean(dim=0, keepdim=True)
        variance  = ((x - mean) ** 2).mean(dim=0)
        mi_scores = variance / (variance.sum() + 1e-8)
        normed    = F.normalize(x, dim=0)
        sim       = normed.T @ normed
        mi_outer  = torch.outer(mi_scores, mi_scores)
        L         = mi_outer * sim
        scores    = torch.diag(L)
        n_keep    = max(1, int(self.keep_rate * x.shape[-1]))
        _, top_idx = torch.topk(scores, n_keep)
        mask = torch.zeros(x.shape[-1], device=x.device)
        mask[top_idx] = 1.0 / self.keep_rate
        return x * mask


class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        device     = features.device
        batch_size = features.shape[0]
        features   = F.normalize(features, dim=1)
        sim_matrix = torch.matmul(features, features.T) / self.temperature
        sim_max, _ = torch.max(sim_matrix, dim=1, keepdim=True)
        sim_matrix = sim_matrix - sim_max.detach()
        labels     = labels.contiguous().view(-1, 1)
        mask_pos   = torch.eq(labels, labels.T).float().to(device)
        mask_self  = torch.eye(batch_size, device=device)
        mask_pos   = mask_pos - mask_self
        exp_sim    = torch.exp(sim_matrix) * (1 - mask_self)
        log_prob   = sim_matrix - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-8)
        num_pos    = mask_pos.sum(dim=1)
        num_pos    = torch.where(num_pos == 0, torch.ones_like(num_pos), num_pos)
        return -(mask_pos * log_prob).sum(dim=1).div(num_pos).mean()


class SimCLRLoss(nn.Module):
    """
    Self-supervised contrastive loss on target domain.
    Two augmented views of same audio = positive pair.
    No labels needed — purely unsupervised.
    """
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, view1, view2):
        batch_size = view1.shape[0]
        view1 = F.normalize(view1, dim=1)
        view2 = F.normalize(view2, dim=1)

        # Concatenate views
        features   = torch.cat([view1, view2], dim=0)  # (2*batch, dim)
        sim_matrix = torch.matmul(features, features.T) / self.temperature

        # Positive pairs: (i, i+batch) and (i+batch, i)
        labels = torch.arange(batch_size, device=view1.device)
        labels = torch.cat([labels + batch_size, labels], dim=0)

        # Remove self-similarity
        mask = torch.eye(2 * batch_size, device=view1.device).bool()
        sim_matrix = sim_matrix.masked_fill(mask, float("-inf"))

        loss = F.cross_entropy(sim_matrix, labels)
        return loss


print("DPPMI, SupCon, SimCLR ready.")

DPPMI, SupCon, SimCLR ready.


In [ ]:
# ── Cell 6: Full Model ─────────────────────────────────────────────────────
class SERModel(nn.Module):
    """
    WavLM-Large (last N layers unfrozen)
    + DPPMI Dropout
    + Projection Head (for contrastive losses)
    + Classifier Head (for CE loss)
    """
    def __init__(self, num_classes, keep_rate=0.87, unfreeze_last_n=6,
                 proj_dim=128):
        super().__init__()

        # ── WavLM backbone ─────────────────────────────────────────────
        self.wavlm = WavLMModel.from_pretrained("microsoft/wavlm-large")

        for param in self.wavlm.parameters():
            param.requires_grad = False

        n_layers = len(self.wavlm.encoder.layers)
        for i, layer in enumerate(self.wavlm.encoder.layers):
            if i >= n_layers - unfreeze_last_n:
                for param in layer.parameters():
                    param.requires_grad = True

        for param in self.wavlm.encoder.layer_norm.parameters():
            param.requires_grad = True

        # ── DPPMI Dropout ──────────────────────────────────────────────
        self.dppmi = DPPMIDropout(keep_rate=keep_rate)

        # ── Projection Head (for contrastive losses) ───────────────────
        self.projection = nn.Sequential(
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, proj_dim)
        )

        # ── Classifier Head ────────────────────────────────────────────
        self.classifier = nn.Sequential(
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def get_features(self, input_values):
        outputs    = self.wavlm(input_values)
        embeddings = outputs.last_hidden_state.mean(dim=1)  # (batch, 1024)
        return embeddings

    def forward(self, input_values):
        embeddings  = self.get_features(input_values)
        sparse      = self.dppmi(embeddings)
        projection  = self.projection(sparse)
        logits      = self.classifier(sparse)
        return logits, projection


# ── Initialize ─────────────────────────────────────────────────────────────
model     = SERModel(
    num_classes    = NUM_CLASSES,
    keep_rate      = KEEP_RATE,
    unfreeze_last_n = UNFREEZE_N,
    proj_dim       = PROJ_DIM
).to(DEVICE)
criterion  = nn.CrossEntropyLoss()
supcon_fn  = SupConLoss(temperature=SUPCON_TEMP).to(DEVICE)
simclr_fn  = SimCLRLoss(temperature=SIMCLR_TEMP).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total params     : {total_params:,}")
print(f"Trainable params : {trainable_params:,}")

config.json:   0%|          | 0.00/2.22k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/488 [00:00<?, ?it/s]

Total params     : 316,701,511
Trainable params : 76,830,999


In [7]:
# ── Cell 7: Training Loop ─────────────────────────────────────────────────
import IPython
display(IPython.display.Javascript('''
function ClickConnect(){
    console.log("Keeping session alive");
    document.querySelector("colab-toolbar-button#connect").click()
}
setInterval(ClickConnect, 60000)
'''))

optimizer = AdamW([
    {"params": [p for p in model.wavlm.parameters() if p.requires_grad], "lr": LR_WAVLM},
    {"params": model.dppmi.parameters(),       "lr": LR_HEAD},
    {"params": model.projection.parameters(),  "lr": LR_HEAD},
    {"params": model.classifier.parameters(),  "lr": LR_HEAD}
], weight_decay=1e-4)

scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=5)


def evaluate(loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for input_values, labels in loader:
            input_values = input_values.float().to(DEVICE)
            logits, _    = model(input_values)
            preds        = logits.argmax(dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    return acc, all_preds, all_labels


def train():
    best_acc       = 0
    patience_count = 0
    history = {
        "ce_loss": [], "supcon_loss": [], "simclr_loss": [],
        "src_acc": [], "tgt_acc": []
    }

    print(f"\n{'='*70}")
    print(f"  WavLM + DPPMI + SupCon(src) + SimCLR(tgt)")
    print(f"{'='*70}")
    print(f"{'Ep':>4} | {'CE':>8} | {'SupCon':>8} | {'SimCLR':>8} | {'Src Acc':>8} | {'Tgt Acc':>8}")
    print("-"*65)

    simclr_iter = iter(tgt_simclr_loader)

    for epoch in range(EPOCHS):
        model.train()
        epoch_ce, epoch_supcon, epoch_simclr = [], [], []

        for src_inputs, src_labels in src_loader:
            src_inputs = src_inputs.float().to(DEVICE)
            src_labels = src_labels.to(DEVICE)

            # Get SimCLR target batch (two augmented views)
            try:
                view1, view2 = next(simclr_iter)
            except StopIteration:
                simclr_iter  = iter(tgt_simclr_loader)
                view1, view2 = next(simclr_iter)
            view1 = view1.float().to(DEVICE)
            view2 = view2.float().to(DEVICE)

            optimizer.zero_grad()

            # Source forward
            src_logits, src_proj = model(src_inputs)
            ce_loss     = criterion(src_logits, src_labels)
            supcon_loss = supcon_fn(src_proj, src_labels)

            # Target forward (two views, no labels)
            _, proj1 = model(view1)
            _, proj2 = model(view2)
            simclr_loss = simclr_fn(proj1, proj2)

            total_loss = (ce_loss
                         + LAMBDA_SUPCON * supcon_loss
                         + LAMBDA_SIMCLR * simclr_loss)

            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            epoch_ce.append(ce_loss.item())
            epoch_supcon.append(supcon_loss.item())
            epoch_simclr.append(simclr_loss.item())

        tgt_acc, _, _ = evaluate(tgt_test_loader)
        if (epoch + 1) % 5 == 0:
            src_acc, _, _ = evaluate(src_loader)
        else:
            src_acc = history["src_acc"][-1] if history["src_acc"] else 0.0

        scheduler.step(tgt_acc)

        history["ce_loss"].append(np.mean(epoch_ce))
        history["supcon_loss"].append(np.mean(epoch_supcon))
        history["simclr_loss"].append(np.mean(epoch_simclr))
        history["src_acc"].append(src_acc)
        history["tgt_acc"].append(tgt_acc)

        print(f"{epoch+1:>4} | "
              f"{np.mean(epoch_ce):>8.4f} | "
              f"{np.mean(epoch_supcon):>8.4f} | "
              f"{np.mean(epoch_simclr):>8.4f} | "
              f"{src_acc*100:>7.2f}% | "
              f"{tgt_acc*100:>7.2f}%")

        if tgt_acc > best_acc:
            best_acc       = tgt_acc
            patience_count = 0
            torch.save(model.state_dict(), SAVE_PATH / "best_wavlm_model.pt")
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print(f"\nEarly stopping at epoch {epoch+1}")
                break

    print(f"\nBest Target Accuracy: {best_acc*100:.2f}%")
    return history, best_acc


history, best_acc = train()

# Save everything
_, final_preds, final_labels = evaluate(tgt_test_loader)
np.save(SAVE_PATH / "preds_wavlm.npy",  np.array(final_preds))
np.save(SAVE_PATH / "labels_wavlm.npy", np.array(final_labels))
with open(SAVE_PATH / "hist_wavlm.json", "w") as f:
    json.dump({k: [float(v) for v in vals] for k, vals in history.items()}, f)
with open(SAVE_PATH / "result_wavlm.json", "w") as f:
    json.dump({"WavLM + DPPMI + SupCon + SimCLR": float(best_acc)}, f, indent=2)

print(f"\nFinal: {best_acc*100:.2f}%")
print("Saved.")

<IPython.core.display.Javascript object>


  WavLM + DPPMI + SupCon(src) + SimCLR(tgt)
  Ep |       CE |   SupCon |   SimCLR |  Src Acc |  Tgt Acc
-----------------------------------------------------------------
   1 |   1.7918 |   2.4378 |   2.4802 |    0.00% |   14.55%
   2 |   1.5153 |   2.3723 |   1.5683 |    0.00% |   28.04%
   3 |   1.3305 |   2.3043 |   1.1088 |    0.00% |   35.27%
   4 |   1.1968 |   2.2448 |   0.9477 |    0.00% |   46.61%
   5 |   1.0977 |   2.2037 |   0.8289 |   50.79% |   45.54%
   6 |   1.0012 |   2.1485 |   0.6396 |   50.79% |   48.66%
   7 |   0.9675 |   2.0875 |   0.4986 |   50.79% |   48.57%
   8 |   0.9273 |   2.0683 |   0.5548 |   50.79% |   45.89%
   9 |   0.8798 |   2.0263 |   0.4686 |   50.79% |   53.57%
  10 |   0.8379 |   1.9618 |   0.5119 |   60.78% |   43.48%
  11 |   0.8022 |   1.9029 |   0.4170 |   60.78% |   51.16%
  12 |   0.7691 |   1.8570 |   0.3432 |   60.78% |   56.07%
  13 |   0.7479 |   1.8230 |   0.3184 |   60.78% |   53.66%
  14 |   0.6784 |   1.7540 |   0.3489 |   60.78% 